In [1]:
from transformers import DetrForObjectDetection, DetrImageProcessor, TrOCRProcessor, VisionEncoderDecoderModel
import torch
import cv2
import supervision as sv
from PIL import Image
import cv2
from time import sleep

2024-08-29 18:28:11.858642: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-29 18:28:13.363635: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
DETR_CHECKPOINT = "/home/ralvarez22/Adrian/Text2Latex/models/detr/Baizhi/V_16"
TROCR_MODEL = "/home/ralvarez22/Documentos/trocr_hand/trocr_llm/finetuned/Akivili/V_5"
DEVICE = "cuda"
CONFIDENCE_TRESHOLD = 0.5
IOU_TRESHOLD = 0.1

In [3]:
detr_proc = DetrImageProcessor.from_pretrained(DETR_CHECKPOINT)
detr_model = DetrForObjectDetection.from_pretrained(
    pretrained_model_name_or_path=DETR_CHECKPOINT, 
    ignore_mismatched_sizes=True, local_files_only=True
).to(DEVICE)

#trocr_proc = TrOCRProcessor.from_pretrained(TROCR_MODEL,device_map=DEVICE)
#trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL, device_map=DEVICE)
# Similar to the TROCR Lab, I force the parameters to avoid mistakes
#trocr_model.generation_config.decoder_start_token_id = trocr_proc.tokenizer.bos_token_id
#trocr_model.generation_config.temperature = 0.1

In [4]:
detr_model.compile()
#trocr_model.compile()

In [5]:
cam = cv2.VideoCapture(0)

In [7]:
while True:
    # Read a frame from the camera
    status, photo = cam.read()
    
    #cv2.imshow("Captured Image", photo)
    # Generate all the Bounding boxes
    with torch.no_grad():
        # load image and predict
        inputs = detr_proc(images=photo, return_tensors='pt').to(DEVICE)
        outputs = detr_model(**inputs)

        # post-process
        target_sizes = torch.tensor([photo.shape[:2]]).to(DEVICE)
        results = detr_proc.post_process_object_detection(
            outputs=outputs, 
            threshold=0.1, 
            target_sizes=target_sizes
        )[0]
    
    scores = results["scores"].tolist()
    bboxes = results["boxes"].tolist()
    pil_image = Image.fromarray(photo).convert("RGB")
    
    for box in bboxes:
        start_point = (int(box[0]), int(box[1]))
        end_point = (int(box[2]), int(box[3]))
        #print(start_point, end_point)
        # draw the rectangle
        cv2.rectangle(photo, start_point, end_point, (0, 0, 255), thickness=1, lineType=cv2.LINE_8) 
        # display the output
    
    cv2.imshow('Annotated Image', photo)
    
    #for idx, e in enumerate(bboxes):
    #    crp_img = pil_image.crop(e)
    #    trocr_pixels = trocr_proc(crp_img, return_tensors="pt").pixel_values.to(DEVICE)
    #    trocr_out = trocr_model.generate(trocr_pixels)
    #    rec_text = trocr_proc.tokenizer.decode(trocr_out[0].cpu(), skip_special_tokens=True)
    #    print("Detected Text: {}".format(rec_text))
    
    # Check for the 'Enter' key press to exit the loop
    if cv2.waitKey(10) == 13:
        break
    
    #sleep(1)
    
cv2.destroyAllWindows()
cam.release()

ValueError: Unable to create tensor, you should probably activate padding with 'padding=True' to have batched tensors with the same length.

: 

In [ ]:
#cv2.destroyAllWindows()
#cam.release()